# Poker solver: CUDA benchmark

1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. **Runtime → Run all**.
3. Copy the output of the last cell (the summary) back to Claude.

The notebook clones the repo, builds the CUDA solver, runs the tests, then times the GPU solver against the CPU solver on the same spots and the same exploitability target.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2
!nproc; free -g | head -2

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y libsqlite3-dev zlib1g-dev > /dev/null && echo deps ok

In [ ]:
!rm -rf pokercfrgpu && git clone -q -b value-net-depth-limit https://github.com/charleszheng0/pokercfrgpu.git && cd pokercfrgpu && git log --oneline -1

In [ ]:
%cd /content/pokercfrgpu
!cmake -S . -B build -DPOKER_CUDA=ON -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j4 --target pokergpu pokerflat tests 2>&1 | grep -E 'error|warning|Built target' ; ls -la build/pokergpu

In [ ]:
!./build/tests --quick 2>&1 | tail -12

In [ ]:
# Same spots, same target, GPU (CUDA) and CPU (depth-first DCFR, this machine's CPU).
!./build/pokergpu --spot river --target 0.3 --step 25 | tee bench_river.txt
!./build/pokergpu --spot turn --target 0.5 --step 20 | tee bench_turn.txt
!./build/pokergpu --spot flop3bet --target 0.5 --step 20 | tee bench_flop3bet.txt

In [ ]:
# Big single-raised flop (about 10.5 GB on the GPU). The CPU run is skipped here: Colab has only a
# few CPU cores, so the CPU baseline comes from the 8-thread laptop.
!./build/pokergpu --spot flopsrp --target 0.5 --step 20 --cpu 0 | tee bench_flopsrp.txt

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader; nproc
!grep -H RESULT bench_*.txt